In [ ]:
import yt_dlp
import os
import cv2
from ultralytics import YOLO

Prompt user to select a file from their system or download a VOD from youtube

In [ ]:
def get_user_choice():
    while True:
        choice = input("Would you like to provide a (1) File Path or (2) YouTube URL Link? Enter 1 or 2: ").strip()
        if choice in ["1", "2"]:
            return choice
        print("Invalid choice. Please enter 1 for File Path or 2 for URL Link.")

def get_file_or_link(choice):
    if choice == "1":
        file_path = input("Enter the file path: ").strip()
        return ("file", file_path)
    elif choice == "2":
        url_link = input("Enter the URL link: ").strip()
        return ("url", url_link)

# Prompt user for choice
user_choice = get_user_choice()
video_type, user_input = get_file_or_link(user_choice)

print(f"You selected a {video_type.upper()} with input: {user_input}")

if video_type == "url": # If user selected URL, download the video from YouTube
    save_folder = "VODS"
    os.makedirs(save_folder, exist_ok=True) # Create the folder VODS to store downloaded videos

    ydl_opts = {
        "format": "bestvideo[height=720]",
        "outtmpl": f"{save_folder}/%(title)s.%(ext)s",  # Save with title as filename
    }

    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        info_dict = ydl.extract_info(user_input, download=True)  # Download URL
        video_title = info_dict.get('title', 'unknown_title').replace(" ", "_")  # Replace spaces with underscores in title name
        video_ext = info_dict.get('ext', 'mp4')  # Get video extension
    
    # Store absolute path of downloaded video
    VOD = os.path.abspath(os.path.join(save_folder, f"{video_title}.{video_ext}"))

elif video_type == "file": # If user selected file, set the VOD to the file path
    VOD = os.path.abspath(user_input)  # Convert user input to absolute path
    print("hello")

# Print the final video path
print(f"Video file path: {VOD}")

You selected a URL with input: https://www.youtube.com/watch?v=In9OAVcg8Dg&t=662s
[youtube] Extracting URL: https://www.youtube.com/watch?v=In9OAVcg8Dg&t=662s
[youtube] In9OAVcg8Dg: Downloading webpage
[youtube] In9OAVcg8Dg: Downloading tv client config
[youtube] In9OAVcg8Dg: Downloading player 7d1d50a6
[youtube] In9OAVcg8Dg: Downloading tv player API JSON
[youtube] In9OAVcg8Dg: Downloading ios player API JSON
[youtube] In9OAVcg8Dg: Downloading m3u8 information
[info] In9OAVcg8Dg: Downloading 1 format(s): 247
[download] Destination: VODS\M80 vs SRB - Challengers NA - Playoffs  - Map 2 Split.webm
[download] 100% of  388.68MiB in 00:00:11 at 34.45MiB/s    
Video file path: c:\Users\ianmk\CODING\val_vision\VODS\M80 vs SRB - Challengers NA - Playoffs  - Map 2 Split.webm


Now we have a variable "VOD" which is a string that represents the system path to the VOD

From here we will

1) Apply the classification model to remove all extra frames (Casting, timeouts, pre-round, etc...)

2) Apply the detection model to output bounding box coords and frame ID

3) Pass the coords and frame ID to OCR model and output data

In [ ]:
# Load the trained YOLO classification model
classify_model = YOLO("runs/classify/train/weights/best.pt")
killfeed_box_model = YOLO("killfeed/results/run21/weights/best.pt")

cap = cv2.VideoCapture(VOD)  # Load the video

batch_size = 10  # Number of frames to process at once
frame_buffer = []
frame_count = 0

# Get video properties
fps = int(cap.get(cv2.CAP_PROP_FPS))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

# Define the killfeed region (x, y, width, height.) 
# fThis is the area our model will check for killfeed boxes.
killfeed_x, killfeed_y, killfeed_w, killfeed_h = 850, 50, 430, 350

while cap.isOpened():
    ret, frame = cap.read()
    
    if not ret:
        break  # Exit when no more frames

    frame_buffer.append(frame)
    frame_count += 1

    if len(frame_buffer) == batch_size:
        # Pass batch of frames to your models
        for f in frame_buffer:
            results = classify_model(frame)
            prediction = results[0].probs.top1 # prediction = 1 if current frame has gameplay, 0 otherwise
            #print("Prediction:", prediction)

            if prediction == 1:
                # NOW WE WANT THE FRAME TO BE PASSED TO THE OBJECT DETECTION MODEL
                # If the detection box is within our killfeed region, we want to pass the frame to the killfeed model
                killfeed_results = killfeed_box_model(f)

                # Step 3: Check if detected objects are inside the killfeed region
                for result in killfeed_results:
                    for box in result.boxes:
                        x_min, y_min, x_max, y_max = box.xyxy[0]  # Get bounding box coordinates

                        # Check if the box is within the killfeed region
                        if (
                            killfeed_x <= x_min <= killfeed_x + killfeed_w and
                            killfeed_y <= y_min <= killfeed_y + killfeed_h
                        ):
                            print(f"Killfeed detected at: {x_min}, {y_min}, {x_max}, {y_max}")
                            # You can save this frame, extract text, or further analyze

        # Clear buffer after processing
        frame_buffer.clear()

# Process remaining frames from video 
if frame_buffer:
    for f in frame_buffer:
        pass  # Replace this with model processing logic

cap.release()
cv2.destroyAllWindows()


